# Comparação V1 vs V2 vs V3

Análise comparativa das três versões do classificador de qualidade de lances de xadrez.
Carrega modelos e artefatos pré-computados das 3 versões — nenhum re-treino é executado.

---

## 1. Setup

In [ ]:
import json
import os
import random
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*", category=UserWarning)
plt.rcParams.update({"font.size": 11, "figure.facecolor": "white"})
sns.set_style("whitegrid")

PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = Path("data")

from version_config import V1, V2, V3, ALL_VERSIONS
from notebook_utils import *

print(f"Diretório de trabalho: {os.getcwd()}")
print("Setup OK")

---

## 2. Carregar Dados e Modelos

Usamos o CSV de features da V3 (superset de todas as features) para ter um X_test comum. Cada modelo recebe apenas as suas features.

In [ ]:
df_features_v3 = pd.read_csv(V3.features_csv)

X_all = df_features_v3.drop(columns=["label"])
y = (df_features_v3["label"] == "ruim").astype(int)

X_temp, X_test, y_temp, y_test = train_test_split(
    X_all, y, test_size=0.15, stratify=y, random_state=RANDOM_SEED
)

print(f"Conjunto de teste: {len(X_test):,} amostras")
print(f"  bom  = {(y_test == 0).sum():,}")
print(f"  ruim = {(y_test == 1).sum():,}")

In [ ]:
dt_v1, rf_v1, names_v1 = V1.load_models()
dt_v2, rf_v2, names_v2 = V2.load_models()
dt_v3, rf_v3, names_v3 = V3.load_models()

print("Modelos carregados:")
for cfg, dt, rf in [(V1, dt_v1, rf_v1), (V2, dt_v2, rf_v2), (V3, dt_v3, rf_v3)]:
    print(f"  {cfg.label}")
    print(f"    DT: max_depth={dt.max_depth}, min_samples_leaf={dt.min_samples_leaf}")
    print(f"    RF: n_estimators={rf.n_estimators}, max_depth={rf.max_depth}")

---

## 3. Tabela Comparativa

Todas as métricas calculadas no mesmo conjunto de teste (16,394 lances).

In [ ]:
rows = []
for cfg, dt, rf, names in [
    (V1, dt_v1, rf_v1, names_v1),
    (V2, dt_v2, rf_v2, names_v2),
    (V3, dt_v3, rf_v3, names_v3),
]:
    X_sub = X_test[names]
    for model_name, model in [("Decision Tree", dt), ("Random Forest", rf)]:
        yp = model.predict(X_sub)
        yproba = model.predict_proba(X_sub)[:, 1]
        rows.append({
            "Versão": f"V{cfg.version}",
            "Modelo": model_name,
            "Accuracy": round(accuracy_score(y_test, yp), 4),
            "F1 (ruim)": round(f1_score(y_test, yp, pos_label=1), 4),
            "Recall (ruim)": round((yp[y_test == 1] == 1).mean(), 4),
            "Precision (ruim)": round((y_test[yp == 1] == 1).mean(), 4) if (yp == 1).any() else 0,
            "ROC-AUC": round(roc_auc_score(y_test, yproba), 4),
        })

df_compare = pd.DataFrame(rows)
df_compare

---

## 4. Deltas por Métrica

Variação entre versões, lidas dos artefatos pré-computados.

In [ ]:
v2_deltas = DATA_DIR / "evaluation_v2" / "v1_vs_v2_deltas.csv"
if v2_deltas.exists():
    print_version_deltas(v2_deltas)

print()

v3_deltas = DATA_DIR / "evaluation_v3" / "v1_v2_v3_deltas.csv"
if v3_deltas.exists():
    print_version_deltas(v3_deltas)

---

## 5. Evolução das Métricas (Barras)

In [ ]:
plot_version_metrics_bars(ALL_VERSIONS, X_test, y_test)

---

## 6. Curvas ROC e Precision-Recall — Overlay

In [ ]:
plot_version_roc_pr_overlay(ALL_VERSIONS, X_test, y_test)

---

## 7. Evolução da Feature #1

| Versão | Feature #1 | Tipo |
|--------|-----------|------|
| V1 | move_number | Contexto |
| V2 | contested_squares | Tática |
| V3 | worst_see_against_player | Look-ahead |

In [ ]:
v3_new_features = [
    "delta_hanging_player", "delta_hanging_opponent", "delta_hanging_value_player",
    "delta_threats_against_player", "delta_mobility_player", "delta_mobility_opponent",
    "delta_contested_squares", "delta_king_attackers_player",
    "opponent_best_capture_value", "opponent_can_check", "opponent_num_good_captures",
    "created_hanging_self", "see_of_move", "worst_see_against_player", "is_losing_capture",
]

plot_top_features_colored(rf_v3, names_v3, names_v1, names_v2, v3_new_features)

print("\nTop 10 features do RF por versão:")
for cfg, rf, names in [(V1, rf_v1, names_v1), (V2, rf_v2, names_v2), (V3, rf_v3, names_v3)]:
    imp = rf.feature_importances_
    idx = np.argsort(imp)[::-1][:5]
    top = [f"{translate(names[i])} ({imp[i]:.4f})" for i in idx]
    print(f"  V{cfg.version}: {', '.join(top)}")

---

## 8. Importância das Features Táticas (V2)

In [ ]:
plot_tactical_features_importance(V2)

---

## 9. Evolução das Regras da Árvore de Decisão

A primeira decisão da árvore evoluiu de descrever o **tipo** do lance para avaliar suas **consequências**:
- **V1:** "É captura?" → descreve a ação
- **V2:** "Casas disputadas > X?" → detecta tensão
- **V3:** "Pior SEE ≤ -0.50?" → avalia consequências de trocas

In [ ]:
for cfg in ALL_VERSIONS:
    print(f"\n{'='*60}")
    print(f"  ÁRVORE DE DECISÃO — V{cfg.version}")
    print(f"{'='*60}")
    print_tree_rules(cfg)
    print()

---

## 10. Síntese: O Ciclo Diagnóstico → Melhoria

| Métrica (RF) | V1 | V2 | V3 | Δ total |
|-------------|-----|-----|-----|--------|
| **Accuracy** | 0.6827 | 0.6912 | 0.7849 | **+10.22 pp** |
| **F1 (ruim)** | 0.3525 | 0.3664 | 0.4312 | **+7.87 pp** |
| **Precision (ruim)** | 0.2589 | 0.2698 | 0.3676 | **+10.87 pp** |
| **Recall (ruim)** | 0.5523 | 0.5710 | 0.5215 | −3.08 pp |
| **ROC-AUC** | 0.6837 | 0.7079 | 0.7678 | **+8.41 pp** |

### O que aprendemos

- **Tipo de informação > quantidade de features**: 15 features V3 trouxeram +6.5pp em F1 vs +1.4pp das 19 features V2.
- **SEE é a feature mais discriminativa**: técnica de 1980s que supera todas as features posicionais e táticas.
- **Trade-off precision/recall**: o RF V3 tem recall menor que V2 mas precision muito maior — modelo mais útil na prática.
- **Meta AUC ≥ 0.75 atingida** (0.7678).
- O ciclo **treino → diagnóstico → melhoria → re-treino** é o resultado central do projeto.